Author: Bryan Sandor

Course: Stat 554

Title: Final Project

# Preamble

For this project, we'll use a `Jupyter` notebook fitting a machine learning model using `pyspark`'s `MLib` module. We'll use code to read in a stream of data and use that model to perform predictions on the stream, outputing them to the console. The data is modified from the UCI machine learning repository and studies the relationship between power consumption in Tetouan City and various factors like time of day, temperature, and humidity.

We begin by importing all the necessary libraries.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml.feature import SQLTransformer, Binarizer

# Part 1: Fitting the Model

## Reading in the Data

We will read the data in using a `pandas` data frame, initially.

In [2]:
powerData = pd.read_csv("power_ml_data.csv")

Then we convert it to a `spark` data frame, first initializing our `spark` session.

In [3]:
spark = SparkSession.builder \
        .appName("proj2") \
        .config("spark.sql.ansi.enabled", "false") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/28 17:32:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/28 17:32:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/28 17:32:36 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
powerDF = spark.createDataFrame(powerData)
powerDF.show(n = 10, truncate = False)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|6.559      |73.8    |0.083     |0.051                |0.119        |34055.6962  |16128.87538 |20240.96386 |1    |0   |
|6.414      |74.5    |0.083     |0.07                 |0.085        |29814.68354 |19375.07599 |20131.08434 |1    |0   |
|6.313      |74.5    |0.08      |0.062                |0.1          |29128.10127 |19006.68693 |19668.43373 |1    |0   |
|6.121      |75.0    |0.083     |0.091                |0.096        |28228.86076 |18361.09422 |18899.27711 |1    |0   |
|5.921      |75.7    |0.081     |0.048                |0.085        |27335.6962  |17872.34043 |18442.40964 |1    |0   |
|5.853      |76.9    |0.081     |0.059  

For this assignment, we will use `Power_Zone_3` as the response and the other variables as predictors.

## Manipulations and Formatting

Now we need to make some adjustments to the data.

### Change `Hour` Class

In [5]:
powerDF.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

First, we need to re-cast the `Hour` column as `DoubleType` instead of `LongType()`.

In [6]:
recast = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS db_hour FROM __THIS__")

powerDF = recast.transform(powerDF)

In [7]:
powerDF

DataFrame[Temperature: double, Humidity: double, Wind_Speed: double, General_Diffuse_Flows: double, Diffuse_Flows: double, Power_Zone_1: double, Power_Zone_2: double, Power_Zone_3: double, Month: bigint, Hour: bigint, db_hour: double]

### Binarize `Hour` Column

Now that the `Hour` column is appropriately cast, we binarize it with the margin at $6.5$, i.e., night vs day.

In [16]:
binaryTrans = Binarizer(threshold = 6.5, inputCol = "db_hour", outputCol = "bin_hour")
binaryTrans.transform(powerDF).show()

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|db_hour|bin_hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-------+--------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|    0.0|     0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|    0.0|     0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|    0.0|     0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|    0.0|     0.0|
|      5.921|    75.7|     0.081|        